# Evidence-Grounded Social Research Agent

**Turn a social research goal into bounded decisions, validated browser evidence, and a source-linked report.**

## Overview

A social research agent has two separate jobs: decide what to do next, then prove what it observed. This tutorial builds that boundary explicitly. A typed decision provider can only select from actions owned by the program; a social adapter returns untrusted cards and post details; an evidence gate validates every source before a report can cite it.

The default example is fully offline and uses four clearly synthetic Instagram records on `example.com`. It demonstrates the control flow, not Jev quality, browser coverage, engagement, or speed. The optional live path uses [Jev Social](https://github.com/socai-io/jev-social), where Jev chooses bounded operations and the local `socai` CLI reads the user's signed-in browser.


## Detailed Explanation

### Motivation

A conventional browser agent can mix planning, navigation, extraction, and prose generation in one opaque loop. That makes three failures hard to distinguish: the model chose the wrong operation, the browser returned incomplete data, or the final answer invented a connection that was never observed.

This agent separates those concerns:

1. **Typed routing:** the decision provider receives only code-owned platform or action choices.
2. **Read-only execution:** the social adapter can search and inspect; it cannot publish, like, follow, message, or download media.
3. **Evidence validation:** URLs, platform identity, text size, record count, and access states are checked before state changes.
4. **Grounded synthesis:** the report iterates over accepted evidence rather than the adapter's raw payload.
5. **Honest outcomes:** complete, partial, and blocked are first-class results.

### Agent Architecture

![Evidence-Grounded Social Research Agent](../images/evidence_grounded_social_research_agent.svg)

### What Jev Changes

Jev is useful at the decision boundary: choose one typed route or one available next operation without asking a generative model to write and then parsing its prose. The surrounding program still owns the allowed choices, confidence gate, step limit, URL policy, and completion rule. The offline provider below mirrors that interface deterministically; it is not Jev and makes no model-quality claim.


## Required Packages

The offline tutorial uses only the Python 3.10+ standard library, so there is nothing to install. The optional live command later requires Node.js 20+, Jev Social v0.1.8, the official `socai` CLI, and an already signed-in Chrome profile. It is intentionally not executed by this notebook.

### Import the standard-library tools


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Protocol, Sequence
from urllib.parse import quote, urlparse
import html
import math
import posixpath
import re
import sys

assert sys.version_info >= (3, 10), "This tutorial requires Python 3.10 or newer."


## Implementation

### 1. Define typed state and adapter contracts

The model-facing surface contains short choice IDs, never shell commands, selectors, or arbitrary URLs. The adapter contract exposes only search and read operations.


In [ ]:
PLATFORMS = ("instagram", "tiktok", "linkedin")
MIN_ROUTE_CONFIDENCE = 0.35
MAX_RECORDS = 10
MAX_TEXT_CHARS = 1_500
MAX_URL_BYTES = 2_048


class DecisionProviderError(RuntimeError):
    """An expected route or action provider failure."""


class AdapterOperationError(RuntimeError):
    """An expected search or read transport failure."""


@dataclass(frozen=True)
class Decision:
    """A typed choice returned by a decision provider."""

    choice: str
    confidence: float


@dataclass(frozen=True)
class Card:
    """A validated discovery card that may be opened for evidence."""

    platform: str
    source_url: str
    preview: str


@dataclass(frozen=True)
class Evidence:
    """One validated observation suitable for citation."""

    platform: str
    source_url: str
    author: str
    text: str
    observed_at: str


@dataclass(frozen=True)
class Action:
    """One code-owned operation offered to the decision provider."""

    name: str
    source_url: str = ""


@dataclass
class ResearchState:
    """Mutable state owned by the orchestration loop."""

    goal: str
    platform: str = ""
    cards: list[Card] = field(default_factory=list)
    evidence: list[Evidence] = field(default_factory=list)
    failed_sources: set[str] = field(default_factory=set)
    trace: list[str] = field(default_factory=list)


@dataclass(frozen=True)
class ResearchResult:
    """A terminal complete, partial, or blocked research result."""

    status: str
    reason: str
    state: ResearchState


class DecisionProvider(Protocol):
    """Interface for a typed route and next-action provider."""

    def choose_route(self, goal: str, choices: Sequence[str]) -> Decision:
        """Choose one platform from the supplied choices."""

    def choose_action(self, choices: Sequence[str]) -> Decision:
        """Choose one available action from the supplied choices."""


class SocialAdapter(Protocol):
    """Read-only boundary for a social data tool such as socai."""

    def search(self, platform: str, query: str, limit: int) -> dict:
        """Return cards or access state; wrap expected I/O failures."""

    def read(self, platform: str, source_url: str) -> dict:
        """Return observed detail; wrap expected I/O failures."""


### 2. Validate every external value

The fixture uses `example.com`; a live adapter must set `allow_fixture=False`, which restricts sources to the selected social platform. The parser also rejects mixed-platform results, duplicated URLs, oversized fields, and unrecognized access states.


In [ ]:
PLATFORM_HOSTS = {
    "instagram": ("instagram.com",),
    "tiktok": ("tiktok.com",),
    "linkedin": ("linkedin.com",),
}
ACCESS_STATES = {"login_required", "challenge_required", "rate_limited"}
CONTROL_CHARS = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
INVALID_PERCENT = re.compile(r"%(?![0-9A-Fa-f]{2})")
PERCENT_OCTET = re.compile(r"%([0-9A-Fa-f]{2})")
UNRESERVED = frozenset("ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789-._~")


def clean_text(value: object, *, limit: int = MAX_TEXT_CHARS) -> str:
    """Normalize untrusted text and enforce a bounded non-empty result."""

    if not isinstance(value, str):
        raise ValueError("expected text")
    normalized = " ".join(CONTROL_CHARS.sub("", value).split())
    if not normalized:
        raise ValueError("text is empty")
    if len(normalized) > limit:
        raise ValueError("text exceeds its field limit")
    return normalized


def canonical_path(raw_path: str) -> str:
    """Normalize unreserved escapes and dot segments for source identity."""

    path = raw_path.replace("\\", "/") or "/"
    if INVALID_PERCENT.search(path):
        raise ValueError("source URL has an invalid percent escape")

    def normalize_octet(match: re.Match[str]) -> str:
        """Decode unreserved octets and uppercase every reserved escape."""

        character = chr(int(match.group(1), 16))
        return character if character in UNRESERVED else f"%{match.group(1).upper()}"

    decoded = PERCENT_OCTET.sub(normalize_octet, path)
    normalized = posixpath.normpath(f"/{decoded.lstrip('/')}")
    return quote(normalized, safe="/%:@-._~!$&'*+,;=")


def validate_source_url(raw: object, platform: str, *, allow_fixture: bool) -> str:
    """Return a normalized HTTPS source URL inside the selected platform policy."""

    if not isinstance(raw, str):
        raise ValueError("source URL must be text")
    if len(raw.encode("utf-8")) > MAX_URL_BYTES:
        raise ValueError("source URL is too long")
    if any(character.isspace() or ord(character) < 32 for character in raw):
        raise ValueError("source URL contains whitespace or control characters")
    parsed = urlparse(raw)
    host = (parsed.hostname or "").lower().rstrip(".")
    if parsed.scheme != "https" or parsed.username or parsed.password or not host:
        raise ValueError("source URL must be credential-free HTTPS")
    try:
        port = parsed.port
    except ValueError as error:
        raise ValueError("source URL has an invalid port") from error
    if port not in (None, 443):
        raise ValueError("source URL uses a non-standard port")
    allowed = host == "example.com" if allow_fixture else any(
        host == suffix or host.endswith(f".{suffix}")
        for suffix in PLATFORM_HOSTS[platform]
    )
    if not allowed:
        raise ValueError(f"source URL is outside the {platform} policy")
    return parsed._replace(
        netloc=host,
        path=canonical_path(parsed.path),
        query="",
        fragment="",
    ).geturl()


def access_state(payload: object) -> str:
    """Extract a recognized access barrier without guessing from free-form text."""

    if not isinstance(payload, dict):
        raise ValueError("adapter payload must be an object")
    status = payload.get("status", "ok")
    if not isinstance(status, str):
        raise ValueError("adapter status must be text")
    if status not in {"ok", "empty", *ACCESS_STATES}:
        raise ValueError("adapter returned an unknown status")
    return status if status in ACCESS_STATES else ""


def parse_cards(
    payload: object,
    platform: str,
    *,
    max_records: int,
    allow_fixture: bool,
) -> list[Card]:
    """Convert a bounded search payload into unique validated cards."""

    if access_state(payload) or payload.get("status") == "empty":
        return []
    results = payload.get("results")
    if not isinstance(results, list):
        raise ValueError("search results must be a list")
    if len(results) > max_records:
        raise ValueError("adapter returned more records than requested")
    cards: list[Card] = []
    seen: set[str] = set()
    for item in results:
        if not isinstance(item, dict) or item.get("platform") != platform:
            raise ValueError("search result has the wrong platform or shape")
        url = validate_source_url(item.get("source_url"), platform, allow_fixture=allow_fixture)
        if url in seen:
            continue
        seen.add(url)
        cards.append(Card(platform, url, clean_text(item.get("preview"), limit=240)))
    return cards


def parse_evidence(payload: object, card: Card, *, allow_fixture: bool) -> Evidence:
    """Validate one opened record and bind it to its discovery card."""

    barrier = access_state(payload)
    if barrier:
        raise PermissionError(barrier)
    if payload.get("status") == "empty":
        raise LookupError("opened record is unavailable")
    if payload.get("platform") != card.platform:
        raise ValueError("opened record changed platform")
    url = validate_source_url(payload.get("source_url"), card.platform, allow_fixture=allow_fixture)
    if url != card.source_url:
        raise ValueError("opened record changed source URL")
    return Evidence(
        platform=card.platform,
        source_url=url,
        author=clean_text(payload.get("author"), limit=80),
        text=clean_text(payload.get("text")),
        observed_at=clean_text(payload.get("observed_at"), limit=40),
    )


### 3. Build deterministic offline adapters

The provider selects only from the choices it receives. The fixture adapter can return success, an empty result, or a platform access barrier so every branch is testable without a browser or account.


In [ ]:
FIXTURE_RECORDS = (
    {
        "platform": "instagram",
        "source_url": "https://example.com/instagram/post-1",
        "preview": "A creator compares two ways to research a video idea.",
        "author": "fixture_creator_1",
        "text": "The creator starts with one narrow audience question and keeps a link beside every note.",
        "observed_at": "2026-09-27T04:00:00Z",
    },
    {
        "platform": "instagram",
        "source_url": "https://example.com/instagram/post-2",
        "preview": "A carousel shows a short evidence review workflow.",
        "author": "fixture_creator_2",
        "text": "The carousel separates discovery cards from posts that were actually opened before writing conclusions.",
        "observed_at": "2026-09-27T04:02:00Z",
    },
    {
        "platform": "instagram",
        "source_url": "https://example.com/instagram/post-3",
        "preview": "A reel describes a small-model routing experiment.",
        "author": "fixture_creator_3",
        "text": "The experiment uses a small decision model for route selection and a separate system for the final explanation.",
        "observed_at": "2026-09-27T04:04:00Z",
    },
    {
        "platform": "instagram",
        "source_url": "https://example.com/instagram/post-4",
        "preview": "A post explains why incomplete samples need labels.",
        "author": "fixture_creator_4",
        "text": "The author marks login barriers and empty searches as coverage limits instead of treating them as negative evidence.",
        "observed_at": "2026-09-27T04:06:00Z",
    },
)


class FixtureDecisionProvider:
    """Deterministic stand-in for Jev's typed decision interface."""

    def __init__(self) -> None:
        """Record offered action IDs so the boundary is executable."""

        self.action_choices: list[tuple[str, ...]] = []

    def choose_route(self, goal: str, choices: Sequence[str]) -> Decision:
        """Select the only platform named in the goal, otherwise abstain softly."""

        lowered = goal.lower()
        matches = [choice for choice in choices if choice in lowered]
        return Decision(matches[0], 0.92) if len(matches) == 1 else Decision(choices[0], 0.20)

    def choose_action(self, choices: Sequence[str]) -> Decision:
        """Prefer search, then unopened records, then a terminal finish action."""

        self.action_choices.append(tuple(choices))
        if "search" in choices:
            return Decision("search", 0.91)
        open_choices = [choice for choice in choices if choice.startswith("open:")]
        return Decision(open_choices[0] if open_choices else "finish", 0.88)


class FixtureSocai:
    """In-memory read-only social adapter with explicit failure modes."""

    def __init__(self, records: Sequence[dict] = FIXTURE_RECORDS, status: str = "ok") -> None:
        """Store synthetic records and an optional access status."""

        if status != "ok" and status not in ACCESS_STATES:
            raise ValueError("unknown fixture status")
        self.records = tuple(records)
        self.status = status

    def search(self, platform: str, query: str, limit: int) -> dict:
        """Return bounded cards or the configured access barrier."""

        if self.status != "ok":
            return {"status": self.status, "results": []}
        matches = [record for record in self.records if record["platform"] == platform]
        return {
            "status": "ok",
            "query": query,
            "results": [
                {key: record[key] for key in ("platform", "source_url", "preview")}
                for record in matches[:limit]
            ],
        }

    def read(self, platform: str, source_url: str) -> dict:
        """Return one synthetic opened record or an explicit empty state."""

        if self.status != "ok":
            return {"status": self.status}
        for record in self.records:
            if record["platform"] == platform and record["source_url"] == source_url:
                return {"status": "ok", **record}
        return {"status": "empty"}


### 4. Run a bounded decision loop

Available actions are rebuilt from validated state on every step. The provider receives only opaque choice IDs—never mutable state or a generic browser tool. The loop completes as soon as the evidence floor is met and validates that the configured step budget can reach it. The step limit bounds completed calls; production providers and adapters must also enforce their own per-call deadlines.


In [ ]:
def validate_decision(decision: object, choices: Sequence[str]) -> Decision:
    """Reject out-of-space choices and malformed confidence values."""

    if not isinstance(decision, Decision):
        raise ValueError("decision provider returned the wrong shape")
    if decision.choice not in choices:
        raise ValueError("decision provider returned an unavailable choice")
    if (
        isinstance(decision.confidence, bool)
        or not isinstance(decision.confidence, (int, float))
        or not math.isfinite(decision.confidence)
        or not 0.0 <= decision.confidence <= 1.0
    ):
        raise ValueError("decision confidence must be between zero and one")
    return decision


def available_actions(state: ResearchState, minimum_evidence: int) -> list[Action]:
    """Build the next read-only action space from validated state."""

    if not state.cards:
        return [Action("search")]
    opened = {item.source_url for item in state.evidence} | state.failed_sources
    actions = [Action("open", card.source_url) for card in state.cards if card.source_url not in opened]
    if len(state.evidence) >= minimum_evidence or not actions:
        actions.append(Action("finish"))
    return actions


def index_actions(actions: Sequence[Action]) -> dict[str, Action]:
    """Map code-owned opaque IDs to actions without exposing source URLs."""

    indexed: dict[str, Action] = {}
    open_index = 0
    for action in actions:
        if action.name == "open":
            choice_id = f"open:{open_index}"
            open_index += 1
        else:
            choice_id = action.name
        if choice_id in indexed:
            raise RuntimeError("duplicate code-owned action ID")
        indexed[choice_id] = action
    return indexed


def run_research(
    goal: str,
    provider: DecisionProvider,
    adapter: SocialAdapter,
    *,
    limit: int = 4,
    minimum_evidence: int = 4,
    max_steps: int = 8,
    allow_fixture: bool = False,
) -> ResearchResult:
    """Execute typed decisions until the run is complete, partial, or blocked."""

    if not 1 <= limit <= MAX_RECORDS:
        raise ValueError("limit is outside the supported range")
    if not 1 <= minimum_evidence <= limit:
        raise ValueError("minimum evidence must fit inside the result limit")
    if not minimum_evidence + 1 <= max_steps <= 30:
        raise ValueError("max steps cannot reach the evidence floor")

    state = ResearchState(goal=clean_text(goal, limit=500))
    try:
        route = validate_decision(provider.choose_route(state.goal, PLATFORMS), PLATFORMS)
    except (DecisionProviderError, OSError, ValueError):
        state.trace.append("reject:invalid_route_decision")
        return ResearchResult("blocked", "decision provider rejected", state)
    state.trace.append(f"route:{route.choice}:{route.confidence:.2f}")
    if route.confidence < MIN_ROUTE_CONFIDENCE:
        return ResearchResult("blocked", "route confidence below policy", state)
    state.platform = route.choice

    for _ in range(max_steps):
        actions = available_actions(state, minimum_evidence)
        by_id = index_actions(actions)
        try:
            decision = validate_decision(provider.choose_action(tuple(by_id)), tuple(by_id))
        except (DecisionProviderError, OSError, ValueError):
            state.trace.append("reject:invalid_action_decision")
            return ResearchResult("blocked", "decision provider rejected", state)
        action = by_id[decision.choice]
        state.trace.append(f"action:{action.name}:{decision.confidence:.2f}")

        if action.name == "search":
            try:
                payload = adapter.search(state.platform, state.goal, limit)
                barrier = access_state(payload)
                if barrier:
                    return ResearchResult("blocked", barrier, state)
                state.cards = parse_cards(
                    payload,
                    state.platform,
                    max_records=limit,
                    allow_fixture=allow_fixture,
                )
            except (AdapterOperationError, OSError):
                state.trace.append("reject:search_failed")
                return ResearchResult("blocked", "adapter search failed", state)
            except ValueError:
                state.trace.append("reject:invalid_payload")
                return ResearchResult("blocked", "adapter payload rejected", state)
            if not state.cards:
                return ResearchResult("partial", "search returned no validated cards", state)
        elif action.name == "open":
            card = next(card for card in state.cards if card.source_url == action.source_url)
            try:
                item = parse_evidence(
                    adapter.read(state.platform, card.source_url),
                    card,
                    allow_fixture=allow_fixture,
                )
            except (AdapterOperationError, OSError):
                state.trace.append("reject:read_failed")
                status = "partial" if state.evidence else "blocked"
                return ResearchResult(status, "adapter read failed", state)
            except PermissionError as error:
                return ResearchResult("blocked", str(error), state)
            except ValueError:
                state.trace.append("reject:invalid_payload")
                return ResearchResult("blocked", "adapter payload rejected", state)
            except LookupError:
                state.failed_sources.add(card.source_url)
                state.trace.append("skip:unavailable")
                continue
            state.evidence.append(item)
            if len(state.evidence) >= minimum_evidence:
                return ResearchResult("complete", "evidence floor reached", state)
        elif action.name == "finish":
            status = "complete" if len(state.evidence) >= minimum_evidence else "partial"
            reason = "evidence floor reached" if status == "complete" else "available cards exhausted"
            return ResearchResult(status, reason, state)

    return ResearchResult("partial", "step limit reached", state)


### 5. Render only accepted evidence

The renderer never receives raw adapter JSON. It reports the terminal status and coverage limit before listing observations, so a partial result cannot look complete.


In [ ]:
def markdown_cell(value: str) -> str:
    """Escape characters that could alter a Markdown table cell."""

    escaped = value.replace("\\", "\\\\")
    for mark in "`*{}[]()#+-.!_|":
        escaped = escaped.replace(mark, f"\\{mark}")
    return html.escape(escaped, quote=True).replace("\n", " ")


def render_report(result: ResearchResult) -> str:
    """Create a source-linked Markdown report from validated evidence only."""

    state = result.state
    lines = [
        "# Social research report",
        "",
        f"- **Status:** {result.status}",
        f"- **Reason:** {result.reason}",
        f"- **Platform:** {state.platform or 'unresolved'}",
        f"- **Validated evidence:** {len(state.evidence)} record(s)",
        "",
    ]
    if not state.evidence:
        lines.append("No source passed the evidence gate; no topical conclusion is reported.")
        return "\n".join(lines)

    lines.extend([
        "## Observed evidence",
        "",
        "| Author | Observation | Source |",
        "| --- | --- | --- |",
    ])
    for item in state.evidence:
        lines.append(
            f"| {markdown_cell(item.author)} | {markdown_cell(item.text)} | "
            f"[open]({item.source_url}) |"
        )
    lines.extend([
        "",
        "## Limits",
        "",
        "This report summarizes only the records opened and validated in this run. "
        "Discovery cards, inaccessible pages, and unobserved comments are not evidence.",
    ])
    return "\n".join(lines)


## Usage Example

### Complete an offline four-source research run

The assertions are executable acceptance checks. They prove that the loop reaches its evidence floor, cites every accepted record, and never exposes an action outside search, open, or finish.


In [ ]:
provider = FixtureDecisionProvider()
fixture = FixtureSocai()
result = run_research(
    "Research creator workflows on Instagram",
    provider,
    fixture,
    limit=4,
    minimum_evidence=4,
    max_steps=8,
    allow_fixture=True,
)
report = render_report(result)

assert result.status == "complete"
assert len(result.state.evidence) == 4
assert report.count("https://example.com/instagram/post-") == 4
assert {entry.split(":", 2)[1] for entry in result.state.trace[1:]} <= {"search", "open", "finish"}
assert provider.action_choices
assert all(
    re.fullmatch(r"(?:search|finish|open:\d+)", choice)
    for offered in provider.action_choices
    for choice in offered
)
assert "&lt;b&gt; &gt; safe" == markdown_cell("<b> > safe")
print(report)


### Exercise blocked, empty, and rejected-input outcomes

A login gate must stop the run without evidence. An empty search is partial, not a claim that the topic does not exist. Malformed decisions, payloads, transport failures, oversized fields, and URL aliases are also exercised as explicit boundary checks.


In [ ]:
blocked = run_research(
    "Research creator workflows on Instagram",
    provider,
    FixtureSocai(status="login_required"),
    allow_fixture=True,
)
empty = run_research(
    "Research creator workflows on Instagram",
    provider,
    FixtureSocai(records=()),
    allow_fixture=True,
)
partial = run_research(
    "Research creator workflows on Instagram",
    provider,
    FixtureSocai(records=FIXTURE_RECORDS[:2]),
    limit=4,
    minimum_evidence=4,
    allow_fixture=True,
)


class InvalidRouteProvider(FixtureDecisionProvider):
    """Return a malformed route value to exercise provider validation."""

    def choose_route(self, goal: str, choices: Sequence[str]) -> Decision:
        """Violate the provider contract deliberately."""

        return None


class MalformedStatusSocai(FixtureSocai):
    """Return a non-string access status for rejection testing."""

    def search(self, platform: str, query: str, limit: int) -> dict:
        """Return an invalid unhashable status."""

        return {"status": []}


class SearchFailureSocai(FixtureSocai):
    """Simulate one expected search transport failure."""

    def search(self, platform: str, query: str, limit: int) -> dict:
        """Raise the adapter's declared operational error."""

        raise AdapterOperationError("synthetic search failure")


class ReadFailureSocai(FixtureSocai):
    """Simulate one expected read transport failure."""

    def read(self, platform: str, source_url: str) -> dict:
        """Raise the adapter's declared operational error."""

        raise AdapterOperationError("synthetic read failure")


invalid_route = run_research(
    "Research creator workflows on Instagram", InvalidRouteProvider(), fixture, allow_fixture=True
)
malformed_status = run_research(
    "Research creator workflows on Instagram", provider, MalformedStatusSocai(), allow_fixture=True
)
search_failure = run_research(
    "Research creator workflows on Instagram", provider, SearchFailureSocai(), allow_fixture=True
)
read_failure = run_research(
    "Research creator workflows on Instagram", provider, ReadFailureSocai(), allow_fixture=True
)

alias_payload = {
    "status": "ok",
    "results": [
        {"platform": "instagram", "source_url": "https://example.com/instagram/post-1", "preview": "one"},
        {"platform": "instagram", "source_url": "https://example.com/a/../instagram/%70ost-1?copy=1", "preview": "duplicate"},
    ],
}
assert len(parse_cards(alias_payload, "instagram", max_records=4, allow_fixture=True)) == 1

try:
    clean_text("x" * (MAX_TEXT_CHARS + 1))
except ValueError:
    pass
else:
    raise AssertionError("oversized evidence text was accepted")

assert (blocked.status, blocked.reason, len(blocked.state.evidence)) == (
    "blocked",
    "login_required",
    0,
)
assert (empty.status, empty.reason) == ("partial", "search returned no validated cards")
assert (partial.status, partial.reason, len(partial.state.evidence)) == (
    "partial",
    "available cards exhausted",
    2,
)
assert (invalid_route.status, invalid_route.reason) == ("blocked", "decision provider rejected")
assert (malformed_status.status, malformed_status.reason) == ("blocked", "adapter payload rejected")
assert (search_failure.status, search_failure.reason) == ("blocked", "adapter search failed")
assert (read_failure.status, read_failure.reason) == ("blocked", "adapter read failed")
print(render_report(blocked))


### Optional live path: Jev × socai

The maintained [Jev Social v0.1.8](https://github.com/socai-io/jev-social/tree/v0.1.8) application implements this boundary against real Instagram, TikTok, and LinkedIn browser evidence. Run it in a terminal, not inside the offline notebook:

~~~bash
npx github:socai-io/jev-social#v0.1.8 onboard
npx github:socai-io/jev-social#v0.1.8 search \
  "find emerging design creators on Instagram" \
  --platform auto --limit 4 --max-steps 12
~~~

The live path is read-only by default but is not offline: Chrome and the selected platform still use the network. With the default configured key, typed decisions and report synthesis use OpenRouter. The documented loopback System One provider replaces only typed decision calls; setting `OPENROUTER_REPORT_MODEL=off` separately selects the deterministic source-linked report. Login, challenge, rate-limit, and empty-result states remain valid outcomes. Review each platform's terms and avoid collecting data you do not need.


## Comparison

| Property | Free-form browser agent | This bounded evidence agent |
| --- | --- | --- |
| Next operation | Generated tool name and arguments | Choice from current code-owned IDs |
| Browser capability | Often broad navigation and interaction | Search and read only |
| External payload | Commonly passed straight to synthesis | Schema, platform, URL, size, and state validated |
| Empty or blocked run | May retry or improvise | Explicit partial or blocked result |
| Completion | Model says it is done | Evidence floor, deterministic exhaustion, and bounded completed steps |
| Report support | Depends on prompt compliance | Renderer sees accepted evidence only |
| Default tutorial cost | Model and browser dependent | Zero; standard-library fixture |

This is a control-flow comparison, not a performance benchmark. A meaningful speed or accuracy comparison needs repeated live runs on the same tasks, browser state, versions, and evidence-quality rubric.


## Additional Considerations

- **Calibrate on your own decisions.** A confidence threshold is a policy only after it has been tested on held-out examples from the real route and action distribution.
- **Do not equate discovery with reading.** A search card can prioritize what to open, but only opened and validated content belongs in the evidence ledger.
- **Preserve source identity.** Canonicalize URLs for deduplication without collapsing distinct repost commentary or deleting the original citation.
- **Treat access barriers as coverage limits.** Login gates, challenges, rate limits, deleted posts, and private accounts do not support a positive or negative topical conclusion.
- **Keep live execution read-only.** Publishing, messaging, liking, following, and media downloads should require separate tools and explicit user intent.
- **Minimize retained data.** Store only fields needed for the research question, redact local paths and secrets, and define a deletion policy for browser artifacts.
- **Separate decision and report providers.** A small typed model can choose operations while a different model—or deterministic code—writes the final report.
- **Make claims reproducible.** Pin application, decision-model, and `socai` versions before publishing timing or coverage results.


## References

- TypeSafe AI, [Introducing System One Models & Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev).
- socai, [local browser automation CLI](https://github.com/socai-io/socai).
- Jev Social, [bounded Jev routing to source-linked social evidence](https://github.com/socai-io/jev-social) and immutable [v0.1.8 source](https://github.com/socai-io/jev-social/tree/v0.1.8).
- OWASP, [LLM Prompt Injection Prevention Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/LLM_Prompt_Injection_Prevention_Cheat_Sheet.html).

Disclosure: this tutorial was contributed by a Jev Social maintainer with help from OpenAI Codex. The offline fixture is synthetic, and no live social account, Jev model, or `socai` browser run was used to produce its example output.
